# Muon Optimizer
### Based on: Keller Jordan et al. (2024) + Moonlight scaling improvements

What this notebook contains -
- Cell 1: Imports
- Cell 2: newton_schulz() — makes singular values uniform
- Cell 3: Muon class — the actual optimizer
- Cell 4: get_muon_and_adamw_params() — splits model params




In [ ]:
import torch
import torch.nn as nn
from typing import Optional

if torch.cuda.is_available():
    print('GPU', torch.cuda.get_device_name(0))


Newton-Schulz Iteration

What this does:
Takes a gradient matrix and returns a version where singular values are uniform.  
This is the core math that makes Muon different from AdamW.

The 3 coefficients (a, b, c):
From Keller Jordan's original implementation. Tuned so the iteration converges in 5 steps.

In [ ]:
def newton_schulz(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    Approximate orthogonalization via Newton-Schulz iteration.
    Takes any 2D matrix G and returns it with more uniform singular values.

    Args:
        G:     Gradient matrix. Shape: (m, n)
        steps: Number of iterations. 5 is enough.
    Returns:
        X: Approximately orthogonal version of G.
    """
    assert G.ndim == 2, 'Newton-Schulz only works on 2D matrices'

    # Coefficients from Keller Jordan's original Muon — tuned for fast convergence
    a, b, c = 3.4445, -4.7750, 2.0315

    # Normalize for numerical stability
    X = G / (G.norm() + 1e-7)

    # Transpose if taller than wide — Newton-Schulz converges faster on wide matrices
    transposed = False
    if X.shape[0] > X.shape[1]:
        X = X.T
        transposed = True

    # Each step makes singular values more uniform
    for _ in range(steps):
        A = X @ X.T
        X = a * X + b * (A @ X) + c * (A @ A @ X)

    if transposed:
        X = X.T

    return X


Muon Optimizer Class

Every training step it does 5 things in order:
1. Compute momentum (running average of gradients)
2. Orthogonalize via Newton-Schulz
3. Scale update to match AdamW magnitude (Moonlight improvement)
4. Apply weight decay
5. Update the parameter


In [ ]:
class Muon(torch.optim.Optimizer):
    """
    Muon: MomentUm Orthogonalized by Newton-Schulz
    Only works on 2D matrix parameters.
    Use AdamW for everything else (embeddings, biases, layernorm).

    Args:
        params:       2D matrix parameters only
        lr:           Learning rate (default 0.02)
        momentum:     Momentum coefficient (default 0.95)
        weight_decay: Regularization (default 0.1)
        ns_steps:     Newton-Schulz iterations (default 5)
        update_scale: Scales update to match AdamW magnitude
    """

    def __init__(
        self,
        params,
        lr: float = 0.02,
        momentum: float = 0.95,
        weight_decay: float = 0.1,
        ns_steps: int = 5,
        update_scale: Optional[float] = 0.3,
    ):
        defaults = dict(
            lr=lr, momentum=momentum, weight_decay=weight_decay,
            ns_steps=ns_steps, update_scale=update_scale,
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self,closure=None):
        for group in self.param_groups:
            lr           = group['lr']
            momentum     = group['momentum']
            weight_decay = group['weight_decay']
            ns_steps     = group['ns_steps']
            update_scale = group['update_scale']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad

                if grad.ndim != 2:
                    raise ValueError(
                        f'Muon received a {grad.ndim}D parameter. '
                        'Only pass 2D matrix parameters to Muon. '
                        'Pass everything else to AdamW.'
                    )

                # Step 1: Momentum buffer
                state = self.state[p]
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(grad)
                buf = state['momentum_buffer']
                buf.mul_(momentum).add_(grad)

                # Step 2: Newton-Schulz orthogonalization
                update = newton_schulz(buf.float(), steps=ns_steps).to(grad.dtype)

                # Step 3: Scale to match AdamW update magnitude
                if update_scale is not None:
                    update_rms = update.norm() / (update.numel() ** 0.5)
                    if update_rms > 1e-7:
                        update = update * (update_scale / update_rms)

                # Step 4: Weight decay
                if weight_decay != 0:
                    update = update + weight_decay * p.data

                # Step 5: Update the parameter
                p.data.add_(update, alpha=-lr)

Parameter Splitter

Goes through every parameter in the model and sorts them:
- 2D matrices - Muon
- Everything else - AdamW

In LoRA (Experiments 1 , 2 & 2b): only LoRA A and B matrices are trainable, so Muon gets exactly those.

In full fine-tuning (Experiment 3): Muon gets all 2D matrices in the whole model.

In [ ]:
def get_muon_and_adamw_params(model: nn.Module):
    """
    Splits model parameters:
    - muon_params:  2D matrix parameters  -> Muon
    - adamw_params: everything else        -> AdamW

    Args:
        model: Any HuggingFace model (with or without LoRA)
    Returns:
        muon_params, adamw_params
    """
    muon_params  = []
    adamw_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:  # skip frozen params
            continue
        if param.ndim == 2:
            muon_params.append(param)
        else:
            adamw_params.append(param)

    return muon_params, adamw_params